# 01 - Data Cleaning

Goal: turn the raw CVM DFP files into a clean table with the key accounts
for the selected retailers, one row per company and year.

**Part 3.1 - Find the CVM code of each company**

We filter companies by their **CVM code** (`CD_CVM`), not by name:
names change over time and vary in spelling, while the code is fixed and unique.

In [1]:
import pandas as pd

RAW_DIR = "../data/raw"  # the notebook runs from inside notebooks/

## Read the company index

`dfp_cia_aberta_<year>.csv` lists every company that filed a DFP that year.
CVM files use `;` as separator and `latin-1` encoding.

In [2]:
index_2025 = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_2025.csv", sep=";", encoding="latin-1")

print(index_2025.shape)
index_2025.head()

(761, 9)


,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,CATEG_DOC,ID_DOC,DT_RECEB,LINK_DOC
0,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DFP,154715,2026-02-11,http://www.rad.cvm.gov.br/ENETCONSULTA/frmDown...
1,00.001.180/0001-26,2025-12-31,1,AXIA ENERGIA S.A.,2437,DFP,154939,2026-02-26,http://www.rad.cvm.gov.br/ENETCONSULTA/frmDown...
2,00.070.698/0001-11,2025-12-31,1,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,14451,DFP,155793,2026-03-27,http://www.rad.cvm.gov.br/ENETCONSULTA/frmDown...
3,00.080.671/0001-00,2025-12-31,1,CARAMURU ALIMENTOS S.A.,26190,DFP,155564,2026-03-20,http://www.rad.cvm.gov.br/ENETCONSULTA/frmDown...
4,00.194.724/0001-13,2025-12-31,1,AUREN OPERAÇÕES S.A.,26174,DFP,155042,2026-03-03,http://www.rad.cvm.gov.br/ENETCONSULTA/frmDown...


In [3]:
index_2025.columns

Index(['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'CATEG_DOC',
       'ID_DOC', 'DT_RECEB', 'LINK_DOC'],
      dtype='str')

## Search the retailers by name

`str.contains` returns a True/False column (a *boolean mask*).
Using the mask inside `[]` keeps only the rows where it is True.
The `|` inside the pattern means "or", so we search the four names at once.

In [4]:
pattern = "MAGAZINE LUIZA|RENNER|SBF|C&A"
mask = index_2025["DENOM_CIA"].str.contains(pattern, case=False)

index_2025[mask][["CD_CVM", "DENOM_CIA", "CNPJ_CIA", "VERSAO", "DT_RECEB"]]

,CD_CVM,DENOM_CIA,CNPJ_CIA,VERSAO,DT_RECEB
159,25410,SBF COMÉRCIO DE PRODUTOS ESPORTIVOS S.A.,06.347.409/0001-65,1,2026-03-09
308,24694,GRUPO SBF S.A.,13.217.485/0001-11,1,2026-03-09
554,24848,C&A MODAS S.A.,45.242.914/0001-05,1,2026-02-24
569,22470,MAGAZINE LUIZA S.A.,47.960.950/0001-21,1,2026-03-12
749,8133,LOJAS RENNER S.A.,92.754.738/0001-62,1,2026-03-05
750,8133,LOJAS RENNER S.A.,92.754.738/0001-62,2,2026-03-05


## Two things to notice

**1. Two "SBF" companies**

- `24694` - **GRUPO SBF S.A.**: the listed holding company (ticker SBFG3).
- `25410` - **SBF COMÉRCIO DE PRODUTOS ESPORTIVOS S.A.**: an operating subsidiary,
  which also has its own CVM registration.

We use **24694**. The holding's *consolidated* statements already include its
subsidiaries, so adding the subsidiary would count the same revenue twice.

**2. Lojas Renner appears twice with the same code**

The rows differ in `VERSAO` (1 and 2): the company re-filed (*reapresentação*)
its DFP. The most recent version is the valid one. We check below whether the
financial statement files also contain both versions.

In [5]:
COMPANIES = {
    22470: "Magazine Luiza",
    8133: "Lojas Renner",
    24694: "Grupo SBF",
    24848: "C&A",
}

## Check: are all companies present in every year?

If a company is missing in some year, our 5-year comparison would have a gap.
`isin` checks whether each value belongs to a list.

In [6]:
for year in range(2021, 2026):
    index_year = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_{year}.csv", sep=";", encoding="latin-1")
    found = index_year[index_year["CD_CVM"].isin(COMPANIES.keys())]["CD_CVM"].unique()
    print(year, "found", len(found), "of", len(COMPANIES))

2021 found 4 of 4
2022 found 4 of 4
2023 found 4 of 4
2024 found 4 of 4
2025 found 4 of 4


## Check: how many versions are in the financial statement files?

`groupby("CD_CVM")` splits the rows by company, and `nunique()` counts the
distinct values of `VERSAO` in each group. If every count is 1, the file
already keeps only the latest version and we don't need to handle it.

In [7]:
for year in range(2021, 2026):
    dre = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_DRE_con_{year}.csv", sep=";", encoding="latin-1")
    dre = dre[dre["CD_CVM"].isin(COMPANIES.keys())]
    print(year, dre.groupby("CD_CVM")["VERSAO"].nunique().to_dict())

2021 {8133: 1, 22470: 1, 24694: 1, 24848: 1}
2022 {8133: 1, 22470: 1, 24694: 1, 24848: 1}
2023 {8133: 1, 22470: 1, 24694: 1, 24848: 1}
2024 {8133: 1, 22470: 1, 24694: 1, 24848: 1}
2025 {8133: 1, 22470: 1, 24694: 1, 24848: 1}


**Conclusion of 3.1:** the four CVM codes are stored in `COMPANIES`.
All companies are present in every year, and the statement files keep
a single (latest) version per company.

---
**Part 3.2 - Load the statements and apply the filters**

We need three statements, all **consolidated** (`_con` files):

| File | Statement |
|---|---|
| `BPA` | Balance sheet - assets |
| `BPP` | Balance sheet - liabilities and equity |
| `DRE` | Income statement |

Filters:
1. Only our four companies (`CD_CVM`).
2. Only the current fiscal year: `ORDEM_EXERC == "ÚLTIMO"`.

Each DFP also brings the **prior year** as a comparative (`PENÚLTIMO`).
If we kept both, every year would appear twice: once as `ÚLTIMO` in its own
file and once as `PENÚLTIMO` in the next year's file.

## Look at one file first

Before writing a loop, it is good practice to open one file and understand it.

In [8]:
dre_2025 = pd.read_csv(f"{RAW_DIR}/dfp_cia_aberta_DRE_con_2025.csv", sep=";", encoding="latin-1")

print(dre_2025.shape)
dre_2025.head()

(31218, 15)


,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
0,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2024-01-01,2024-12-31,3.01,Receitas de Intermediação Financeira,273505274.0,S
1,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.01,Receitas de Intermediação Financeira,319462104.0,S
2,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2024-01-01,2024-12-31,3.01.01,Receita de Juros,273505274.0,N
3,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2025-01-01,2025-12-31,3.01.01,Receita de Juros,319462104.0,N
4,00.000.000/0001-91,2025-12-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2024-01-01,2024-12-31,3.02,Despesas de Intermediação Financeira,-168990827.0,S


In [9]:
# How many rows of each fiscal year order are in the file?
dre_2025["ORDEM_EXERC"].value_counts()

ORDEM_EXERC
PENÚLTIMO    15609
ÚLTIMO       15609
Name: count, dtype: int64

In [10]:
# One company, current year only: this is the income statement as published
magalu_2025 = dre_2025[
    (dre_2025["CD_CVM"] == 22470) & (dre_2025["ORDEM_EXERC"] == "ÚLTIMO")
]

magalu_2025[["CD_CONTA", "DS_CONTA", "VL_CONTA", "ESCALA_MOEDA"]]

,CD_CONTA,DS_CONTA,VL_CONTA,ESCALA_MOEDA
21999,3.01,Receita de Venda de Bens e/ou Serviços,3.870339e+07,MIL
22001,3.02,Custo dos Bens e/ou Serviços Vendidos,-2.714884e+07,MIL
22003,3.03,Resultado Bruto,1.155455e+07,MIL
22005,3.04,Despesas/Receitas Operacionais,-9.635868e+06,MIL
22007,3.04.01,Despesas com Vendas,-7.375871e+06,MIL
22009,3.04.02,Despesas Gerais e Administrativas,-2.677056e+06,MIL
22011,3.04.02.01,Gerais e Administrativas,-1.392269e+06,MIL
22013,3.04.02.02,Depreciacao,-1.284787e+06,MIL
22015,3.04.03,Perdas pela Não Recuperabilidade de Ativos,-4.558190e+05,MIL
22017,3.04.04,Outras Receitas Operacionais,7.456100e+05,MIL


## A function to load and filter one statement for one year

The same steps repeat for 3 statements x 5 years = 15 files,
so we write them once inside a function.

In [11]:
def load_statement(statement, year):
    """Read one consolidated statement file and keep our companies, current year only."""
    path = f"{RAW_DIR}/dfp_cia_aberta_{statement}_con_{year}.csv"
    df = pd.read_csv(path, sep=";", encoding="latin-1")

    is_our_company = df["CD_CVM"].isin(COMPANIES.keys())
    is_current_year = df["ORDEM_EXERC"] == "ÚLTIMO"
    df = df[is_our_company & is_current_year].copy()

    df["STATEMENT"] = statement
    df["YEAR"] = year
    return df

In [12]:
STATEMENTS = ["BPA", "BPP", "DRE"]
YEARS = range(2021, 2026)

frames = []
for statement in STATEMENTS:
    for year in YEARS:
        frames.append(load_statement(statement, year))

statements = pd.concat(frames, ignore_index=True)

print(statements.shape)
statements.head()

(4246, 17)


,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA,STATEMENT,YEAR,DT_INI_EXERC
0,13.217.485/0001-11,2021-12-31,1,GRUPO SBF S.A.,24694,DF Consolidado - Balanço Patrimonial Ativo,REAL,MIL,ÚLTIMO,2021-12-31,1,Ativo Total,7146492.0,S,BPA,2021,NaN
1,13.217.485/0001-11,2021-12-31,1,GRUPO SBF S.A.,24694,DF Consolidado - Balanço Patrimonial Ativo,REAL,MIL,ÚLTIMO,2021-12-31,1.01,Ativo Circulante,3617346.0,S,BPA,2021,NaN
2,13.217.485/0001-11,2021-12-31,1,GRUPO SBF S.A.,24694,DF Consolidado - Balanço Patrimonial Ativo,REAL,MIL,ÚLTIMO,2021-12-31,1.01.01,Caixa e Equivalentes de Caixa,549347.0,S,BPA,2021,NaN
3,13.217.485/0001-11,2021-12-31,1,GRUPO SBF S.A.,24694,DF Consolidado - Balanço Patrimonial Ativo,REAL,MIL,ÚLTIMO,2021-12-31,1.01.02,Aplicações Financeiras,0.0,S,BPA,2021,NaN
4,13.217.485/0001-11,2021-12-31,1,GRUPO SBF S.A.,24694,DF Consolidado - Balanço Patrimonial Ativo,REAL,MIL,ÚLTIMO,2021-12-31,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0,S,BPA,2021,NaN


## Checks

1. Every company should have rows for every statement and year (no gaps).
2. The fiscal year should end on 31 December for every company and year,
   otherwise the years would not be comparable.

In [13]:
# Rows per company, statement and year. A zero would mean a gap.
statements.groupby(["CD_CVM", "STATEMENT", "YEAR"]).size().unstack("YEAR")

YEAR              2021  2022  2023  2024  2025
CD_CVM STATEMENT                              
8133   BPA          66    66    66    66    68
       BPP         116   118   118   118   118
       DRE          34    34    34    34    34
22470  BPA          64    64    64    64    64
       BPP         109   111   111   111   111
       DRE          32    32    32    32    32
24694  BPA          64    64    64    64    64
       BPP         112   114   114   114   115
       DRE          30    30    30    30    30
24848  BPA          66    67    67    68    68
       BPP         109   109   112   112   114
       DRE          38    38    39    39    38

In [14]:
# Fiscal year end date for each company and year
statements.groupby(["CD_CVM", "YEAR"])["DT_FIM_EXERC"].unique().unstack("YEAR")

YEAR,2021,2022,2023,2024,2025
CD_CVM,,,,,
8133,[2021-12-31],[2022-12-31],[2023-12-31],[2024-12-31],[2025-12-31]
22470,[2021-12-31],[2022-12-31],[2023-12-31],[2024-12-31],[2025-12-31]
24694,[2021-12-31],[2022-12-31],[2023-12-31],[2024-12-31],[2025-12-31]
24848,[2021-12-31],[2022-12-31],[2023-12-31],[2024-12-31],[2025-12-31]


**Conclusion of 3.2:** `statements` holds the consolidated BPA, BPP and DRE of the
four companies for 2021-2025, current fiscal year only, with no gaps.

---
**Part 3.3 - Convert all values to reais**

`ESCALA_MOEDA` tells the unit of `VL_CONTA`:
- `MIL`: thousands of reais
- `UNIDADE`: reais

We create a column `VALUE` with every amount in reais, so companies that
report in different scales can be compared directly.

In [15]:
# Which scales appear in our data?
statements["ESCALA_MOEDA"].value_counts()

ESCALA_MOEDA
MIL    4246
Name: count, dtype: int64

In [16]:
SCALE_FACTOR = {"UNIDADE": 1, "MIL": 1000}

statements["SCALE_FACTOR"] = statements["ESCALA_MOEDA"].map(SCALE_FACTOR)

# Fail fast: a scale that is not in the dictionary becomes NaN after map()
if statements["SCALE_FACTOR"].isna().any():
    unknown = statements[statements["SCALE_FACTOR"].isna()]["ESCALA_MOEDA"].unique()
    raise ValueError(f"Unknown scale(s) in ESCALA_MOEDA: {unknown}")

statements["VALUE"] = statements["VL_CONTA"] * statements["SCALE_FACTOR"]

In [17]:
# Sanity check: Magazine Luiza net revenue (account 3.01) in 2025
check = statements[
    (statements["CD_CVM"] == 22470)
    & (statements["YEAR"] == 2025)
    & (statements["CD_CONTA"] == "3.01")
]

check[["DS_CONTA", "VL_CONTA", "ESCALA_MOEDA", "VALUE"]]

,DS_CONTA,VL_CONTA,ESCALA_MOEDA,VALUE
4180,Receita de Venda de Bens e/ou Serviços,38703387.0,MIL,3.870339e+10


**Conclusion of 3.3:** `VALUE` holds every amount in reais.
From now on we use `VALUE` instead of `VL_CONTA`.

---
**Part 3.4 - Map the accounts and build the final table**

We only use **fixed accounts** (`ST_CONTA_FIXA == "S"`): they belong to the
standard CVM chart of accounts, so the same code means the same thing for
every company. Company-specific sub-accounts (`"N"`) are not comparable.

| Code | CVM description | Our name |
|---|---|---|
| `1` | Ativo Total | `total_assets` |
| `1.01` | Ativo Circulante | `current_assets` |
| `2.01` | Passivo Circulante | `current_liabilities` |
| `2.02` | Passivo Não Circulante | `noncurrent_liabilities` |
| `2.03` | Patrimônio Líquido Consolidado | `equity` |
| `3.01` | Receita de Venda de Bens e/ou Serviços | `net_revenue` |
| `3.03` | Resultado Bruto | `gross_profit` |
| `3.05` | Resultado Antes do Resultado Financeiro e dos Tributos | `operating_income` |
| `3.11` | Lucro/Prejuízo Consolidado do Período | `net_income` |

Notes:
- `2.02` is not in the original list, but we need it for the debt ratio in step 4
  (total liabilities = current + non-current).
- `2.03` and `3.11` both **include non-controlling interests**, so ROE
  (net income / equity) compares two consistent figures.

In [18]:
ACCOUNTS = {
    "1": "total_assets",
    "1.01": "current_assets",
    "2.01": "current_liabilities",
    "2.02": "noncurrent_liabilities",
    "2.03": "equity",
    "3.01": "net_revenue",
    "3.03": "gross_profit",
    "3.05": "operating_income",
    "3.11": "net_income",
}

selected = statements[statements["CD_CONTA"].isin(ACCOUNTS.keys())].copy()
selected["ACCOUNT"] = selected["CD_CONTA"].map(ACCOUNTS)

selected[["CD_CVM", "YEAR", "CD_CONTA", "DS_CONTA", "ACCOUNT", "VALUE"]].head(10)

,CD_CVM,YEAR,CD_CONTA,DS_CONTA,ACCOUNT,VALUE
0,24694,2021,1,Ativo Total,total_assets,7.146492e+09
1,24694,2021,1.01,Ativo Circulante,current_assets,3.617346e+09
64,24848,2021,1,Ativo Total,total_assets,8.672666e+09
65,24848,2021,1.01,Ativo Circulante,current_assets,3.929398e+09
130,22470,2021,1,Ativo Total,total_assets,3.838462e+10
131,22470,2021,1.01,Ativo Circulante,current_assets,2.450981e+10
194,8133,2021,1,Ativo Total,total_assets,2.141198e+10
195,8133,2021,1.01,Ativo Circulante,current_assets,1.398478e+10
260,24694,2022,1,Ativo Total,total_assets,8.470247e+09
261,24694,2022,1.01,Ativo Circulante,current_assets,4.526488e+09


## Checks before reshaping

1. Every selected account must be a fixed account.
2. Every company and year must have all 9 accounts, no more, no less.

In [19]:
if not (selected["ST_CONTA_FIXA"] == "S").all():
    raise ValueError("Some selected accounts are not fixed CVM accounts")

accounts_per_company_year = selected.groupby(["CD_CVM", "YEAR"]).size()
if not (accounts_per_company_year == len(ACCOUNTS)).all():
    raise ValueError("Some company/year is missing accounts or has duplicates")

print("OK:", len(accounts_per_company_year), "company-years with", len(ACCOUNTS), "accounts each")

OK: 20 company-years with 9 accounts each


## Reshape: one row per company and year

Right now the data is in **long** format: one row per account.
For the ratios we want **wide** format: one row per company and year,
one column per account. `pivot` does this reshape.

In [20]:
financials = selected.pivot(index=["CD_CVM", "YEAR"], columns="ACCOUNT", values="VALUE")
financials = financials.reset_index()

financials["COMPANY"] = financials["CD_CVM"].map(COMPANIES)

columns_order = ["COMPANY", "CD_CVM", "YEAR"] + list(ACCOUNTS.values())
financials = financials[columns_order]
financials.columns.name = None  # remove the leftover "ACCOUNT" label

financials

,COMPANY,CD_CVM,YEAR,total_assets,current_assets,current_liabilities,noncurrent_liabilities,equity,net_revenue,gross_profit,operating_income,net_income
0,Lojas Renner,8133,2021,2.141198e+10,1.398478e+10,7.954100e+09,3.651264e+09,9.806621e+09,1.057156e+10,6.172378e+09,8.139440e+08,6.331120e+08
1,Lojas Renner,8133,2022,2.114889e+10,1.305377e+10,7.005579e+09,4.055798e+09,1.008752e+10,1.327114e+10,7.984814e+09,1.407401e+09,1.291704e+09
2,Lojas Renner,8133,2023,2.049064e+10,1.219164e+10,7.492553e+09,2.950864e+09,1.004722e+10,1.364785e+10,8.220602e+09,8.890770e+08,9.762590e+08
3,Lojas Renner,8133,2024,2.036459e+10,1.231666e+10,7.647557e+09,1.944082e+09,1.077295e+10,1.443637e+10,8.741949e+09,1.259343e+09,1.196668e+09
4,Lojas Renner,8133,2025,1.962603e+10,1.163376e+10,6.866734e+09,2.303012e+09,1.045628e+10,1.582946e+10,9.767882e+09,1.797160e+09,1.457566e+09
5,Magazine Luiza,22470,2021,3.838462e+10,2.450981e+10,1.525719e+10,1.186620e+10,1.126123e+10,3.527815e+10,8.486805e+09,4.706020e+08,5.906610e+08
6,Magazine Luiza,22470,2022,3.776584e+10,2.193814e+10,1.483292e+10,1.228422e+10,1.064870e+10,3.729900e+10,1.043890e+10,7.716350e+08,-4.989750e+08
7,Magazine Luiza,22470,2023,3.745507e+10,2.022116e+10,1.740813e+10,1.043641e+10,9.610534e+09,3.676815e+10,1.016388e+10,-3.721480e+08,-9.791040e+08
8,Magazine Luiza,22470,2024,3.731186e+10,1.955082e+10,1.671055e+10,9.282046e+09,1.131926e+10,3.803807e+10,1.162726e+10,1.562432e+09,4.487170e+08
9,Magazine Luiza,22470,2025,3.791116e+10,1.983905e+10,1.718279e+10,9.450336e+09,1.127803e+10,3.870339e+10,1.155455e+10,1.918683e+09,2.046030e+08


## Accounting check: Assets = Liabilities + Equity

If the mapping is right, the balance sheet must balance for every row.

In [21]:
liabilities_and_equity = (
    financials["current_liabilities"]
    + financials["noncurrent_liabilities"]
    + financials["equity"]
)
difference = financials["total_assets"] - liabilities_and_equity

print("Largest difference (R$):", difference.abs().max())

Largest difference (R$): 0.0


## Easier to read: values in R$ millions

In [22]:
value_columns = list(ACCOUNTS.values())

in_millions = financials.copy()
in_millions[value_columns] = (in_millions[value_columns] / 1_000_000).round(0)

in_millions

,COMPANY,CD_CVM,YEAR,total_assets,current_assets,current_liabilities,noncurrent_liabilities,equity,net_revenue,gross_profit,operating_income,net_income
0,Lojas Renner,8133,2021,21412.0,13985.0,7954.0,3651.0,9807.0,10572.0,6172.0,814.0,633.0
1,Lojas Renner,8133,2022,21149.0,13054.0,7006.0,4056.0,10088.0,13271.0,7985.0,1407.0,1292.0
2,Lojas Renner,8133,2023,20491.0,12192.0,7493.0,2951.0,10047.0,13648.0,8221.0,889.0,976.0
3,Lojas Renner,8133,2024,20365.0,12317.0,7648.0,1944.0,10773.0,14436.0,8742.0,1259.0,1197.0
4,Lojas Renner,8133,2025,19626.0,11634.0,6867.0,2303.0,10456.0,15829.0,9768.0,1797.0,1458.0
5,Magazine Luiza,22470,2021,38385.0,24510.0,15257.0,11866.0,11261.0,35278.0,8487.0,471.0,591.0
6,Magazine Luiza,22470,2022,37766.0,21938.0,14833.0,12284.0,10649.0,37299.0,10439.0,772.0,-499.0
7,Magazine Luiza,22470,2023,37455.0,20221.0,17408.0,10436.0,9611.0,36768.0,10164.0,-372.0,-979.0
8,Magazine Luiza,22470,2024,37312.0,19551.0,16711.0,9282.0,11319.0,38038.0,11627.0,1562.0,449.0
9,Magazine Luiza,22470,2025,37911.0,19839.0,17183.0,9450.0,11278.0,38703.0,11555.0,1919.0,205.0


## Save the clean table

`data/processed/` is ignored by Git, like `data/raw/`: anyone can rebuild
this file by running the download script and this notebook.

In [23]:
financials.to_csv("../data/processed/financials.csv", index=False)
print("Saved", financials.shape[0], "rows")

Saved 20 rows


**Conclusion of 3.4:** `financials` has one row per company and year (4 x 5 = 20 rows),
with the 9 accounts in reais, ready for the ratios in step 4.